# Automated AI Bakery Verification

This notebook will use the procedure developed in the bakery-verification branch to more strictly apply a bakery definition and filter bakery candidates from our intial model using Gemini API. The bakery candidates will be processed in descending ranks and each verification will be saved and considered for the overall verification after some manual review. Businesses will be processed in API batches and the process will be resumed across multiple days due to API rate limits. Previous verification results are loaded automatically so that businesses with completed verification are not resubmitted. The results will later be included in a final bakery name classification results csv.

In [28]:
from pathlib import Path
from getpass import getpass
from datetime import datetime

import json
import time
import pandas as pd

from google import genai
from google.genai import types, errors

# Verification settings
MODEL = "gemini-2.5-flash"

BUSINESSES_PER_REQUEST = 10
MAX_REQUESTS_PER_RUN = 500
REQUEST_DELAY_SECONDS = 2

# Paths
FHRS_DATA_DATE = "2026-07-23"

DATA_FOLDER = Path("../data/business/interim")
VERIFICATION_FOLDER = (DATA_FOLDER / "ai_verification_v2")
VERIFICATION_FOLDER.mkdir(parents=True, exist_ok=True)

RANKED_FHRS_PATH = (DATA_FOLDER / f"london_fhrs_ranked_establishments_{FHRS_DATA_DATE}.csv")
RAW_FHRS_PATH = Path(f"../data/business/raw/london_fhrs_raw_{FHRS_DATA_DATE}.csv")

RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_results.csv")
BATCH_LOG_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_batches.jsonl")
ERROR_LOG_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_errors.jsonl")

# Loading dataset and checking recall range

Since verification via Gemini API can be costly, I found that choosing a recall threshold based on the classifier findings would be a more reasonable way to verify the business names.

In [18]:
fhrs_ranked = pd.read_csv(RANKED_FHRS_PATH)

address_columns = ["FHRSID", "AddressLine1", "AddressLine2", "AddressLine3", "AddressLine4"]

fhrs_raw_address = pd.read_csv(RAW_FHRS_PATH, usecols=address_columns)

fhrs_raw_address = (fhrs_raw_address.drop_duplicates(subset="FHRSID"))

fhrs_ranked = fhrs_ranked.merge(fhrs_raw_address, on="FHRSID", how="left", validate="many_to_one")

fhrs_ranked["Address"] = fhrs_ranked[["AddressLine1", "AddressLine2", "AddressLine3", "AddressLine4"]
                                     ].apply(lambda row: ", ".join(str(value).strip()
                                             for value in row 
                                             if pd.notna(value) and str(value).strip()), 
                                             axis=1)

verification_candidates = (
    fhrs_ranked[fhrs_ranked["BakeryRank"].notna()]
    .sort_values(["BakeryRank", "FHRSID"])
    .drop_duplicates(subset="BusinessNameClean")
    .rename(columns={"FHRSID": "FHRSIDRep"})
    .reset_index(drop=True))

# Candidate-selection threshold from classifier cross-validation (notebook_07)
RECALL_THRESHOLDS = {
    0.90: 0.021,
    0.95: 0.015,
    0.99: 0.006}

threshold_summary = pd.DataFrame({
    "TargetRecall": RECALL_THRESHOLDS.keys(),
    "BakeryScoreThreshold": RECALL_THRESHOLDS.values()
})

threshold_summary["Candidates"] = [(verification_candidates["BakeryScore"] >= threshold).sum()
                                   for threshold in threshold_summary["BakeryScoreThreshold"]]

threshold_summary["CandidateIncreaseFrom90%"] = ((
    threshold_summary["Candidates"] / (threshold_summary.loc[threshold_summary["TargetRecall"] == 0.90,"Candidates"].iloc[0]) - 1
    )* 100).round(2)

threshold_summary["RecallGainFrom90%"] = (threshold_summary["TargetRecall"] - 0.90) * 100

display(threshold_summary)

,TargetRecall,BakeryScoreThreshold,Candidates,CandidateIncreaseFrom90%,RecallGainFrom90%
0,0.90,0.021,19902,0.00,0.0
1,0.95,0.015,28847,44.95,5.0
2,0.99,0.006,49147,146.95,9.0


From this we can see that from 90% recall to 95% recall we add ~45% more candidates for potenitally 5% improvement in recall. Meanwhile for 9% increase from 90% recall to 99% recall we would add ~147% of candidates, which seems unreasonable as we are more than doubling the candidates for a potential improvement in recall

In [19]:
TARGET_RECALL = 0.95
MIN_BAKERY_SCORE = RECALL_THRESHOLDS[TARGET_RECALL]

verification_queue = (verification_candidates[verification_candidates["BakeryScore"] >= MIN_BAKERY_SCORE]
    .sort_values("BakeryRank")
    .reset_index(drop=True)
)

print(f"""
Selected target recall: {TARGET_RECALL:.0%}
BakeryScore threshold: {MIN_BAKERY_SCORE}
Businesses to verify: {len(verification_queue)}""")


Selected target recall: 95%
BakeryScore threshold: 0.015
Businesses to verify: 28847


In [29]:
if RESULTS_PATH.exists():
    previous_results = pd.read_csv(RESULTS_PATH)
    completed_names = set(previous_results["BusinessNameClean"].dropna())

else:
    previous_results = pd.DataFrame()
    completed_names = set()

remaining_queue = verification_queue[~verification_queue["BusinessNameClean"].isin(completed_names)].copy()
remaining_queue = (remaining_queue.sort_values("BakeryRank").reset_index(drop=True))

max_businesses_per_run = (BUSINESSES_PER_REQUEST * MAX_REQUESTS_PER_RUN)

run_queue = (remaining_queue
             .head(max_businesses_per_run)
             .copy()
             .reset_index(drop=True))

requests_planned = (len(run_queue) + BUSINESSES_PER_REQUEST- 1) // BUSINESSES_PER_REQUEST

print(
f"""Previously completed: {len(completed_names)}
Remaining businesses: {len(remaining_queue)}
Businesses selected this run: {len(run_queue)}
Maximum API requests this run: {requests_planned}""")

Previously completed: 18452
Remaining businesses: 10395
Businesses selected this run: 5000
Maximum API requests this run: 500


In [16]:
api_key = getpass("Gemini API key: ")

client = genai.Client(api_key=api_key)

In [21]:
def clean_value(value):
    if pd.isna(value):
        return "Unknown"
    return str(value)

def build_verification_prompt(batch):

    batch = batch.reset_index(drop=True)

    business_blocks = []

    for i, row in batch.iterrows():
        business_blocks.append(
            f"""
BUSINESS {i + 1}
Name: {clean_value(row["BusinessName"])}
Address: {clean_value(row["Address"])}
Postcode: {clean_value(row["PostCode"])}
Local authority: {clean_value(row["LocalAuthorityName"])}
FHRS type: {clean_value(row["BusinessType"])}
""".strip()
        )

    business_text = "\n\n".join(business_blocks)

    prompt = f"""
ROLE

You are verifying food businesses for a dissertation measuring physical retail
bakery provision in London.

Use Google Search to identify each supplied FHRS business and determine whether
it represents a genuine physical bakery location under the definition below.

TASK

For EACH business:

1. Identify the exact business represented by the supplied FHRS record.
2. Check whether the business appears to be currently active at that location.
3. Determine whether customers can physically visit a shop, bakery, cafe,
   market stall or other customer-facing premises to buy products.
4. Determine whether qualifying bakery products are a core, specialist or
   meaningful part of what customers can buy there.
5. Return the required fields using the exact output format below.

Assess each business independently.

Use UNCLEAR when reliable evidence is genuinely insufficient.
Do not guess or invent certainty.

CONTEXT

The FHRS data is dated 23 July 2026.

The purpose of this classification is to map physical bakery provision across
London.

The relevant unit is therefore a customer-facing physical retail location,
not simply any business that produces something baked.

BAKERY DEFINITION

A BAKERY is a physical customer-facing business where qualifying bakery
products are a core, specialist or meaningful part of the products sold at
that location.

Qualifying bakery products can include:

- breads and rolls;
- pastries and viennoiserie;
- pies and baked savoury goods;
- biscuits and cookies;
- doughnuts;
- bagels;
- traditional or culturally specific baked breads, pastries and savouries;
- similar products normally associated with bakery retail.

A business does NOT need to sell many different categories of bakery product.

A specialist physical shop can qualify even when it mainly sells one bakery
product category.

Examples include:

- a bread bakery specialising in sourdough;
- a physical doughnut shop selling many varieties of doughnuts;
- a cookie shop specialising in cookies;
- a bagel bakery;
- a pie shop;
- a patisserie;
- a bakery-cafe;
- a chain bakery outlet.

Specialising in one bakery-product category is not by itself a reason for
exclusion.

The definition is not limited to British or European bakery traditions.

South Asian, Middle Eastern, Eastern European, African and other culturally
specific businesses can qualify when baked breads, pastries, savoury baked
goods or comparable bakery products are an important part of the physical
retail offering.

CAKE-ONLY BUSINESSES

Do NOT classify a business as BAKERY merely because it makes or sells cakes.

Businesses primarily focused on bespoke, celebration, wedding, novelty or
made-to-order cakes should normally be NOT_BAKERY when they do not operate as
a broader bakery or patisserie.

This includes:

- Instagram cake sellers;
- home-based cake makers;
- online cake businesses;
- bespoke cake studios;
- celebration-cake businesses;
- physical cake boutiques whose retail offering is essentially cakes only.

A physical bakery or patisserie can still qualify when cakes are sold alongside
a meaningful wider bakery offering such as pastries, breads, viennoiserie,
cookies or other qualifying bakery products.

PHYSICAL RETAIL REQUIREMENT

Return PHYSICAL YES only when reliable evidence indicates that customers can
visit a physical premises or regular physical retail stall associated with the
supplied business and buy products there.

Examples can include:

- a bakery shop;
- a patisserie;
- a bakery-cafe;
- a physical specialist cookie, doughnut, bagel or bread shop;
- a regular customer-facing market stall;
- another genuine physical retail premises.

Return PHYSICAL NO when reliable evidence positively indicates that the
business operates only or primarily through:

- a private or residential home;
- online orders;
- social-media orders;
- delivery;
- collection from a private residential address;
- a production kitchen with no customer-facing retail outlet; or
- wholesale supply with no customer-facing retail premises.

Do NOT infer PHYSICAL NO merely because a physical shop cannot be found.

Absence of evidence is not evidence that a business is home-based or
online-only.

If reliable evidence is insufficient to determine whether customers can visit
and purchase from a physical location, return PHYSICAL UNCLEAR.

A website, Instagram page, Facebook page, delivery service, Companies House
registration, postal address or map pin does not by itself prove that a
customer-facing retail premises exists.

BAKERY FOCUS REQUIREMENT

Return FOCUS YES when qualifying bakery products are a core, specialist,
defining or meaningful part of the customer-facing business.

Return FOCUS NO when:

- bakery products are merely incidental or minor;
- another type of food business clearly dominates the offering;
- the business is a cake-only business under the cake rule above; or
- reliable evidence shows that it does not meaningfully operate as bakery
  retail.

Do NOT infer FOCUS NO merely because detailed product information cannot be
found.

If reliable evidence is insufficient to determine the importance of bakery
products, return FOCUS UNCLEAR.

CAFES AND MIXED FOOD BUSINESSES

Do NOT automatically classify a cafe as NOT_BAKERY.

A cafe can qualify when bakery products are a substantial, characteristic or
meaningful part of what it sells, even when it also sells drinks, breakfasts,
sandwiches or other food.

A cafe should be NOT_BAKERY when bakery products appear to be only incidental
to a general cafe or restaurant menu.

When the importance of the bakery offering cannot reasonably be determined,
return FOCUS UNCLEAR rather than guessing.

Businesses that should normally be FOCUS NO include:

- ordinary restaurants where bakery products are incidental;
- supermarkets, grocery shops and convenience stores;
- general dessert parlours;
- sweet shops and chocolatiers without a meaningful bakery offering;
- caterers;
- pizza restaurants or takeaways;
- hotels, pubs, schools, nurseries and care homes;
- cake-decorating or baking-supply businesses.

LOCATION MATCH

Return LOCATION YES when the external evidence reasonably identifies the
supplied FHRS business.

A matching full address or postcode is strong evidence.

When FHRS provides no street address or only a partial postcode, a distinctive
business name operating in the same London borough or postcode area can support
LOCATION YES when there is no conflicting location evidence.

Return LOCATION UNCLEAR when:

- multiple plausible businesses match the name;
- the name is too generic to identify the correct business;
- location evidence conflicts with the FHRS information; or
- reliable evidence cannot connect the external business to the supplied
  record.

Do not use a similarly named business elsewhere in London or the UK as evidence
for the supplied record.

For chain businesses, evidence should support the supplied branch or location,
not merely establish that the wider brand operates bakeries.

CURRENT STATUS

Return STATUS ACTIVE when reliable current evidence indicates that the supplied
business is operating at the relevant location.

Return STATUS INACTIVE when reliable evidence indicates that it is permanently
closed, dissolved, dormant or no longer operating at the relevant location.

Return STATUS UNCLEAR when current status cannot reasonably be determined.

Inactive businesses do not represent current bakery provision.

CLASSIFICATION

Choose BAKERY only when ALL of the following are supported:

- LOCATION = YES;
- STATUS = ACTIVE;
- PHYSICAL = YES; and
- FOCUS = YES.

Choose NOT_BAKERY when the correct business is identified and reliable evidence
establishes at least one clear reason for exclusion, including:

- STATUS = INACTIVE;
- PHYSICAL = NO; or
- FOCUS = NO.

Choose UNCLEAR when:

- the business cannot reliably be identified;
- physical retail presence cannot reasonably be determined;
- bakery focus cannot reasonably be determined; or
- available evidence is otherwise insufficient for a supported decision.

Do not classify from the business name or FHRS business type alone.

SOURCE RESTRICTIONS

The supplied FHRS record is candidate-identification information.

It is NOT evidence that the business satisfies the bakery definition.

Food Standards Agency/FHRS pages, local-authority food-hygiene pages, Scores on
the Doors and websites that merely reproduce food-hygiene records must NOT be
used as evidence of:

- physical customer-facing retail;
- bakery focus;
- current customer access; or
- products currently sold.

These sources may only help corroborate the supplied business name, address or
identity.

Likewise, a postal address, Companies House registration, map pin or search
result title alone does not prove that customers can visit and purchase bakery
products there.

SEARCH AND EVIDENCE

Search for each business separately.

Use enough searches to establish:

- business identity;
- current status;
- physical retail presence; and
- actual products or business activity.

If the first search is inconclusive, try an alternative search using the
postcode, address, local authority or distinctive business name before
returning UNCLEAR.

Prefer evidence in this order where available:

1. official business website or current menu;
2. official business social-media page;
3. recognised ordering or delivery platform;
4. Companies House information;
5. credible directories, news reports or other business listings.

Use evidence describing what the business actually sells and how customers
access it.

Specific evidence about products and premises is stronger than a broad company
description, SIC code, FHRS business type or business name.

A delivery platform can provide evidence of products and current business
activity but does not by itself prove that walk-in retail is available.

Companies House can help identify a business or establish company status but
does not by itself prove physical retail access or bakery focus.

Do not continue searching once reliable evidence is sufficient for a supported
decision.

Once a decisive reason for exclusion is established, remaining fields may stay
UNCLEAR when further searching would not change the verdict.

For example:

- if the correct business is confirmed as permanently inactive, PHYSICAL and
  FOCUS may remain UNCLEAR;
- if reliable evidence proves that the business is online-only or home-based,
  further searching for bakery focus is unnecessary.

EXAMPLES

Example A:

A physical bakery shop sells sourdough loaves, focaccia, pastries and other
baked goods directly to customers.

→ YES | ACTIVE | YES | YES | BAKERY


Example B:

A physical patisserie sells pastries, viennoiserie and cakes from a retail shop.

→ YES | ACTIVE | YES | YES | BAKERY


Example C:

A physical cookie shop sells a large range of cookies directly to walk-in
customers and cookies are the defining product of the business.

→ YES | ACTIVE | YES | YES | BAKERY


Example D:

A physical doughnut shop specialises in many varieties of doughnuts sold
directly to customers.

→ YES | ACTIVE | YES | YES | BAKERY


Example E:

A South Asian bakery has a physical shop selling breads, savoury baked goods
and baked sweets directly to customers.

→ YES | ACTIVE | YES | YES | BAKERY


Example F:

A cafe has a physical premises and sells coffee and meals, but also has a
substantial bakery and pastry offering that is a characteristic part of the
business.

→ YES | ACTIVE | YES | YES | BAKERY


Example G:

A general cafe has a physical premises but bakery products consist only of a
small incidental selection alongside its main food and drink menu.

→ YES | ACTIVE | YES | NO | NOT_BAKERY


Example H:

An Instagram business makes bespoke celebration cakes from a residential
address and offers delivery or private collection.

→ YES | ACTIVE | NO | NO | NOT_BAKERY


Example I:

A physical cake boutique specialises only in bespoke wedding and celebration
cakes and does not offer a broader bakery or patisserie range.

→ YES | ACTIVE | YES | NO | NOT_BAKERY


Example J:

A wholesale bakery manufactures breads and pastries for other businesses but
customers cannot visit a retail outlet at the supplied location.

→ YES | ACTIVE | NO | YES | NOT_BAKERY


Example K:

The correct bakery business is identified, but reliable evidence cannot
determine whether customers can visit a physical retail premises.

→ YES | ACTIVE | UNCLEAR | YES | UNCLEAR


Example L:

Several unrelated businesses match a generic name and partial postcode, so the
supplied FHRS record cannot be identified reliably.

→ UNCLEAR | UNCLEAR | UNCLEAR | UNCLEAR | UNCLEAR


Example M:

The correct business is identified but reliable evidence shows that it has
permanently closed.

→ YES | INACTIVE | UNCLEAR | UNCLEAR | NOT_BAKERY


OUTPUT

Return exactly {len(batch)} lines.

Use exactly this format:

business number | location | status | physical | focus | verdict | short evidence reason

Allowed values:

location: YES or UNCLEAR
status: ACTIVE, INACTIVE or UNCLEAR
physical: YES, NO or UNCLEAR
focus: YES, NO or UNCLEAR
verdict: BAKERY, NOT_BAKERY or UNCLEAR

Example output:

1 | YES | ACTIVE | YES | YES | BAKERY | Official website shows a physical bakery shop selling breads and pastries directly to customers.
2 | YES | ACTIVE | NO | NO | NOT_BAKERY | Official page describes bespoke cakes made from a residential address for delivery and collection.
3 | UNCLEAR | UNCLEAR | UNCLEAR | UNCLEAR | UNCLEAR | Generic name and partial postcode match several businesses with no reliable identification evidence.

Keep each reason to ONE short sentence of no more than 25 words.

Return every business number from 1 to {len(batch)} exactly once.

FINAL CHECK

Before returning each line, check that the verdict is consistent with the
returned fields.

PHYSICAL YES requires positive external evidence of customer-facing physical
retail. An FHRS address, business type, company registration or business name
alone is not sufficient.

PHYSICAL NO also requires positive external evidence that customers cannot buy
from a physical retail premises or regular retail stall. An FHRS business type,
a registered/private address, or simply failing to find evidence of a shop is
not sufficient. If that is all the evidence available, return PHYSICAL UNCLEAR.

BAKERY requires:

YES | ACTIVE | YES | YES

NOT_BAKERY requires LOCATION YES and at least one clear exclusion:

STATUS INACTIVE, PHYSICAL NO, or FOCUS NO.

Otherwise return UNCLEAR.

Do not invent certainty simply to avoid UNCLEAR.

Based on all criteria above, evaluate the supplied businesses.

BUSINESSES

{business_text}
""".strip()

    return prompt

In [22]:
def verify_current_batch(batch):

    prompt = build_verification_prompt(batch)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            temperature=0,
            max_output_tokens=1500,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0
            )
        )
    )

    return prompt, response

In [23]:
def get_grounding_info(response):

    if not response.candidates:
        return False, [], [], 0

    grounding = response.candidates[0].grounding_metadata

    if grounding is None:
        return False, [], [], 0

    search_queries = list(grounding.web_search_queries or [])

    source_urls = []

    for chunk in grounding.grounding_chunks or []:
        if chunk.web:
            source_urls.append(chunk.web.uri)

    source_urls = list(dict.fromkeys(source_urls))

    support_count = len(grounding.grounding_supports or [])

    grounding_used = bool(search_queries)

    return (grounding_used, search_queries, source_urls, support_count)

In [24]:
def parse_response(response_text, batch, batch_id):

    batch = batch.reset_index(drop=True)

    text = response_text.replace("\\_", "_").strip()

    # Gemini sometimes forgets newlines between results
    for number in range(len(batch), 0, -1):
        text = text.replace(f"{number} |", f"\n{number} |")

    parsed = {}

    for line in text.splitlines():
        parts = [part.strip()
                for part in line.split("|", 6)]

        if len(parts) != 7:
            continue

        number_text = (parts[0].replace("*", "").strip())

        if not number_text.isdigit():
            continue

        business_number = int(number_text)

        if not 1 <= business_number <= len(batch):
            continue

        location_match = parts[1].upper()
        status = parts[2].upper()
        physical_retail = parts[3].upper()
        bakery_focus = parts[4].upper()
        verdict = (parts[5].upper().replace(" ", "_"))
        reason = parts[6]

        if location_match not in {"YES", "UNCLEAR"}:
            continue

        if status not in {"ACTIVE", "INACTIVE", "UNCLEAR"}:
            continue

        if physical_retail not in {"YES", "NO", "UNCLEAR"}:
            continue

        if bakery_focus not in {"YES", "NO", "UNCLEAR"}:
            continue

        if verdict not in { "BAKERY", "NOT_BAKERY", "UNCLEAR"}:
            continue

        # BAKERY should require every inclusion condition
        if verdict == "BAKERY":
            if not (location_match == "YES"
                    and status == "ACTIVE"
                    and physical_retail == "YES"
                    and bakery_focus == "YES"):
                        continue

        # If identity is unclear, the final verdict must also be unclear
        if (location_match == "UNCLEAR" and verdict != "UNCLEAR"):
            continue

        # NOT_BAKERY needs at least one clear exclusion reason
        if verdict == "NOT_BAKERY":

            clear_exclusion = (status == "INACTIVE"
                               or physical_retail == "NO"
                               or bakery_focus == "NO")
            
            if (location_match != "YES" or not clear_exclusion):
                continue

        row = batch.iloc[business_number - 1]

        parsed[business_number] = {
            "BusinessNameClean": row["BusinessNameClean"],
            "BusinessName": row["BusinessName"],
            "FHRSIDRep": row["FHRSIDRep"],
            "BusinessType": row["BusinessType"],
            "PostCode": row["PostCode"],
            "LocalAuthorityName": row["LocalAuthorityName"],
            "Address": row["Address"],
            "BakeryRank": row["BakeryRank"],
            "BakeryScore": row["BakeryScore"],
            "StoreCount": row["StoreCount"],

            "LocationMatch": location_match,
            "AIStatus": status,
            "PhysicalRetail": physical_retail,
            "BakeryFocus": bakery_focus,
            "AIVerdict": verdict,
            "AIReason": reason,

            "BatchID": batch_id,
            "Model": MODEL,
            "VerificationDateTime": datetime.now().isoformat(timespec="seconds")
        }

    missing_numbers = [number
                       for number in range(1, len(batch) + 1)
                       if number not in parsed]

    results = pd.DataFrame([parsed[number]
                            for number in sorted(parsed)])

    return results, missing_numbers


def save_results(results):

    if results.empty:
        return

    results.to_csv(RESULTS_PATH, mode="a", header=not RESULTS_PATH.exists(), index=False)


def save_jsonl(path, record):

    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False, default=str)+ "\n")

In [25]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
requests_attempted = 0
results_saved = 0

for start in range(0, len(run_queue), BUSINESSES_PER_REQUEST):

    batch = (run_queue.iloc[start:start + BUSINESSES_PER_REQUEST]
             .copy()
             .reset_index(drop=True))

    requests_attempted += 1
    batch_id = f"{RUN_ID}_{requests_attempted}"

    try:
        prompt, response = verify_current_batch(batch)

        finish_reason = None
        finish_message = None

        if response.candidates:
            finish_reason = str(response.candidates[0].finish_reason)
            finish_message = (response.candidates[0].finish_message)

        grounding_used, search_queries, source_urls, support_count = (get_grounding_info(response))

        # Verification without grounding is not accepted
        if not grounding_used:
            save_jsonl(ERROR_LOG_PATH, {
                "BatchID": batch_id,
                "ErrorType": "NO_SEARCH_GROUNDING",
                "Businesses": batch["BusinessName"].tolist()})

            print(f"Batch {requests_attempted}: no Google Search grounding - not saved")

            time.sleep(REQUEST_DELAY_SECONDS)
            continue

        results, missing_numbers = (parse_response(response.text, batch, batch_id))

        # Save successful grounded results
        save_results(results)
        results_saved += len(results)

        # Save the complete API request evidence
        save_jsonl(BATCH_LOG_PATH, {
            "BatchID": batch_id,
            "Time": datetime.now().isoformat(timespec="seconds"),
            "BusinessCount": len(batch),
            "ParsedCount": len(results),
            "BusinessNames": batch["BusinessName"].tolist(),
            "RawResponse": response.text,
            "GroundingUsed": grounding_used,
            "SearchQueries": search_queries,
            "SourceURLs": source_urls,
            "GroundingSupportCount": support_count,
            "Model": MODEL,
            "FinishReason": finish_reason,
            "FinishMessage": finish_message})

        # Record anything Gemini failed to return properly
        if missing_numbers:
            missing_businesses = [batch.iloc[number - 1]["BusinessName"]
                                  for number in missing_numbers]

            save_jsonl(ERROR_LOG_PATH, {
                "BatchID": batch_id,
                "ErrorType": "INCOMPLETE_RESPONSE",
                "Businesses": missing_businesses})

        print(f"Batch {requests_attempted}: saved {len(results)}/{len(batch)}")

    except errors.APIError as error:

        save_jsonl(ERROR_LOG_PATH, {
            "BatchID": batch_id,
            "ErrorType": "API_ERROR",
            "ErrorCode": error.code,
            "ErrorMessage": error.message,
            "Businesses": batch["BusinessName"].tolist()})

        print(f"Batch {requests_attempted} failed: {error.code}")

        if error.code == 429:
            print("Rate limit reached. Stopping safely.")
            break

        if 400 <= error.code < 500:
            print("Non-retryable API error. Stopping safely.")
            break

    except Exception as error:

        save_jsonl(ERROR_LOG_PATH, {
            "BatchID": batch_id,
            "ErrorType": type(error).__name__,
            "ErrorMessage": str(error),
            "Businesses": batch["BusinessName"].tolist()})

        print(f"Batch {requests_attempted} failed: {error}")

    time.sleep(REQUEST_DELAY_SECONDS)

Batch 1: saved 10/10
Batch 2: saved 10/10
Batch 3: saved 10/10
Batch 4: saved 10/10
Batch 5: saved 10/10
Batch 6: saved 10/10
Batch 7: saved 10/10
Batch 8: saved 10/10
Batch 9: saved 10/10
Batch 10: saved 10/10
Batch 11: saved 10/10
Batch 12: saved 10/10
Batch 13: saved 10/10
Batch 14: saved 10/10
Batch 15: saved 10/10
Batch 16: saved 10/10
Batch 17: saved 10/10
Batch 18: saved 10/10
Batch 19: saved 10/10
Batch 20: saved 10/10
Batch 21: saved 10/10
Batch 22: saved 10/10
Batch 23: saved 10/10
Batch 24: saved 10/10
Batch 25: saved 10/10
Batch 26: saved 10/10
Batch 27: saved 10/10
Batch 28: saved 10/10
Batch 29: saved 10/10
Batch 30: saved 10/10
Batch 31: saved 10/10
Batch 32: saved 10/10
Batch 33: saved 10/10
Batch 34: saved 10/10
Batch 35: saved 10/10
Batch 36: saved 10/10
Batch 37: saved 10/10
Batch 38: saved 10/10
Batch 39: saved 10/10
Batch 40: saved 10/10
Batch 41: saved 10/10
Batch 42: saved 10/10
Batch 43: saved 10/10
Batch 44: saved 10/10
Batch 45: saved 10/10
Batch 46: saved 10/

KeyboardInterrupt: 

In [27]:
if RESULTS_PATH.exists():
    all_results = pd.read_csv(RESULTS_PATH)

    print(
        f"""Requests attempted this run: {requests_attempted}
Results saved this run: {results_saved}
Total businesses completed: {len(all_results)}""")

    print(f"\nVerdicts:\n{all_results['AIVerdict'].value_counts()}")

    print(f"\nStatus:\n{all_results['AIStatus'].value_counts()}")

    print(f"\nPhysical retail:\n{all_results['PhysicalRetail'].value_counts()}")

    print(f"\nBakery focus:\n{all_results['BakeryFocus'].value_counts()}")

Requests attempted this run: 405
Results saved this run: 3985
Total businesses completed: 18452

Verdicts:
AIVerdict
NOT_BAKERY    11886
UNCLEAR        4997
BAKERY         1569
Name: count, dtype: int64

Status:
AIStatus
ACTIVE      13350
UNCLEAR      4569
INACTIVE      533
Name: count, dtype: int64

Physical retail:
PhysicalRetail
NO         6807
YES        6254
UNCLEAR    5391
Name: count, dtype: int64

Bakery focus:
BakeryFocus
NO         10288
UNCLEAR     6312
YES         1852
Name: count, dtype: int64
